# NLP Pipeline Experiments

This notebook demonstrates the end‑to‑end NLP pipeline used by Ingredia. It walks through the steps of normalisation, tokenisation, exact matching, alias lookup and fuzzy matching.

The notebook is designed to run locally or in Google Colab – simply install the repository dependencies and adjust the `DATA_ROOT` path to point at your CSV/JSON examples. No modifications to the backend code or database are performed.

In [ ]:
# Install required packages (Colab)
!pip install -q pydantic fastapi sqlalchemy pillow pytesseract rapidfuzz

In [ ]:
# Adjust PYTHONPATH so imports resolve when running in Colab or a fresh checkout
import sys, os
sys.path.append(os.path.abspath('..'))  # assume notebooks are one level below project root

In [ ]:
# Import the NLP components from the backend package
from app.nlp.normalizer import normalize_text
from app.nlp.synonym_matcher import SynonymIndex
from app.nlp.fuzzy_matcher import FuzzyMatcher
from app.nlp.ingredient_linker import IngredientLinker
# Helper to load a small manual test set – adjust as needed
import csv, pathlib

def load_examples(csv_path: str):
    examples = []
    with open(csv_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            examples.append({
                'raw': row['raw_text'],
                'expected': [int(i) for i in row['expected_ids'].split(';') if i]
            })
    return examples

DATA_ROOT = pathlib.Path('data')
EXAMPLES_CSV = DATA_ROOT / 'nlp_example_cases.csv'
if EXAMPLES_CSV.exists():
    test_cases = load_examples(str(EXAMPLES_CSV))
else:
    test_cases = []
    print('No example CSV found at', EXAMPLES_CSV)

## Initialise the linking service

The `IngredientLinker` builds the exact/alias index from the current database and configures a fuzzy matcher with the thresholds defined in `app.config`.


In [ ]:
from app.services.matching_service import get_linker
linker = get_linker()  # FastAPI dependency replacement – creates a fresh linker

## Function to run the full pipeline on a raw string

It normalises, tokenises, matches each token and returns a structured summary.

In [ ]:
def run_pipeline(text: str, linker: IngredientLinker):
    norm = normalize_text(text)
    tokens = norm.split(',')  # simple split on commas for demo
    results = []
    for token in tokens:
        token = token.strip()
        if not token:
            continue
        match = linker.link_ingredient(token)
        results.append({
            'raw_token': token,
            'normalized': match.normalized_token,
            'ingredient_id': match.ingredient_id,
            'method': match.match_method,
            'confidence': match.match_confidence,
            'requires_review': match.requires_review,
            'fuzzy_candidates': [c.dict() for c in match.fuzzy_candidates]
        })
    return results

## Run on the manual examples (if any) and report simple statistics

In [ ]:
if test_cases:
    correct = 0
    total = 0
    for case in test_cases:
        pipeline_res = run_pipeline(case['raw'], linker)
        matched_ids = [r['ingredient_id'] for r in pipeline_res if r['ingredient_id'] is not None]
        total += 1
        if set(matched_ids) == set(case['expected']):
            correct += 1
    print(f'Exact/alias match accuracy on {total} examples: {correct}/{total} ({correct/total:.2%})')
else:
    print('No manual examples to evaluate – add a CSV at data/nlp_example_cases.csv')

## Fuzzy matching thresholds

The fuzzy matcher uses Jaro‑Winkler scores. The configuration in `app.config` defines two thresholds:
* `fuzzy_auto_accept_threshold` – scores above this are automatically accepted (rare).
* `fuzzy_suggest_threshold` – scores above this are presented as suggestions needing user review.

We display the thresholds here for reference.

In [ ]:
from app.config import get_settings
settings = get_settings()
print('Auto‑accept threshold:', settings.fuzzy_auto_accept_threshold)
print('Suggest threshold:', settings.fuzzy_suggest_threshold)

## Ambiguous and unresolved cases

Tokens that fall below the suggestion threshold are marked as `unresolved`. They appear with `requires_review=True` and no `ingredient_id`.

Below we demonstrate a string that yields ambiguous results.

In [ ]:
ambiguous_text = 'Coco‑Beta‑Glucoside, UnknownIngredient'
amb_res = run_pipeline(ambiguous_text, linker)
for r in amb_res:
    print(r)

## Findings (to be filled after running the notebook)

* **Exact/alias accuracy** – summary printed above.
* **Fuzzy behavior** – note how many candidates appear and their scores.
* **Unresolved tokens** – count of tokens that need manual review.

These observations help tune the thresholds in `app.config` and improve the synonym database.